# 1. Pydantic高级特性

上一节用 Pydantic 定义了最简单的输出结构，本节介绍更多写法：

- 1.1–1.6：可选字段、默认值、枚举、列表、嵌套、限制条件
- 1.7：同一个 Pydantic 类，为什么 GPT 和 DeepSeek 对默认值、限制条件的处理不同
- 1.8：其他常用的高级特性：自定义校验器、`Union`、`model_dump`
- 1.9：总结

In [1]:
import asyncio
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

## 1.1 可选字段
问题：LLM 未填充某些字段怎么办？

使用 Optional 指定字段为可选的。

In [2]:
from pydantic import BaseModel, Field
from typing import Optional

class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

structured_llm = model.with_structured_output(Person)
structured_llm.invoke("张三是一名医生")

Person(name='张三', age=0, occupation='医生')

作为对比：

In [3]:
from pydantic import BaseModel, Field
from typing import Optional

class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")

structured_llm = model.with_structured_output(Person)
structured_llm.invoke("张三是一名医生")

Person(name='张三', age=None, occupation='医生')

## 1.2 默认值

LLM 未提供的信息会使用默认值。

格式如下：

Field(default="默认值", description="描述")

先用 DeepSeek 试试，原文没有提到年龄：

In [7]:
from typing import Optional
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import BaseModel,Field

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(default=32,description="年龄")
    occupation: str = Field(description="职业")


# 除了连接信息，什么参数都不传，看看默认值
model_deepseek = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)

structured_llm = model_deepseek.with_structured_output(Person)
structured_llm.invoke("张三是一名医生")

Person(name='张三', age=32, occupation='医生')

同样的 `Person` 类，换成 GPT（`model`）：

In [10]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)
structured_llm = model.with_structured_output(Person)
structured_llm.invoke("张三是一名医生")

Person(name='张三', age=0, occupation='医生')

DeepSeek 用上了默认值 32，GPT 却返回了 `age=0`。同一个 Pydantic 类，为什么结果不同？原因见 1.7。

## 1.3 枚举类型

问题：如何限制字段的可选值？

回答：使用枚举。


举例1：

In [ ]:
from enum import Enum
# Enum (枚举)：表示 Priority 类里的所有成员都是一组固定的、不应该被修改的常量（例如：高、中、低优先级）。
# str (字符串)：表示这个枚举里的每个成员的值必须是字符串类型。
class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class Task(BaseModel):
    title: str
    priority: Priority # 只能是 LOW/MEDIUM/HIGH

class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")

# 测试
structured_llm = model.with_structured_output(CustomerInfo)

conversation = """客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下"""
result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)

print("\n提取结果：")
print(f" 客户: {result.name}")
print(f" 电话: {result.phone}")
print(f" 邮箱: {result.email or '未提供'}")
print(f" 问题: {result.issue}")
print(f" 紧急程度: {result.urgency.value}")

name='王小明' phone='138-1234-5678' email=None issue='订单一直没发货，客户很着急。' urgency=<Priority.HIGH: '高'>

提取结果：
 客户: 王小明
 电话: 138-1234-5678
 邮箱: 未提供
 问题: 订单一直没发货，客户很着急。
 紧急程度: 高


如果嫌单独定义一个 Enum 类太麻烦，也可以直接导入 typing 中的 Literal ，直接在字段里把允许
的值写死。

In [ ]:
from typing import Optional, Literal
from pydantic import BaseModel, Field
class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field("未提供", description="邮箱")
    issue: str = Field(description="问题描述")
    # 使用 Literal 直接限定字面量值
    # 但是复用性较差
    urgency: Literal["低","中","高"] = Field(description="紧急程度")

# 测试
structured_llm = model.with_structured_output(CustomerInfo)

conversation = """客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下"""
result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)
print("\n提取结果：")
print(f" 客户: {result.name}")
print(f" 电话: {result.phone}")
print(f" 邮箱: {result.email}")
print(f" 问题: {result.issue}")
print(f" 紧急程度: {result.urgency}")

name='王小明' phone='138-1234-5678' email=None issue='订单一直没发货' urgency='高'

提取结果：
 客户: 王小明
 电话: 138-1234-5678
 邮箱: None
 问题: 订单一直没发货
 紧急程度: 高


注意 `email` 的默认值是"未提供"，GPT 返回的却是 `None`。这和 1.2 是同一个原因，见 1.7.2。

## 1.4 列表提取

举例1：

In [13]:
from typing import List
class Person(BaseModel):
    """人物信息"""
    name: str
    age: int
class PersonList(BaseModel):
    """人物列表信息"""
    people: List[Person] # 多个 Person 对象
structured_llm = model.with_structured_output(PersonList)
result = structured_llm.invoke("张三 30岁，李四 25岁")
print(result)

people=[Person(name='张三', age=30), Person(name='李四', age=25)]


举例2:

In [14]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")

structured_llm = model.with_structured_output(Review)
review = structured_llm.invoke("""
iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。""")
print(review)

product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格较贵', '不附带充电器']


## 1.5 嵌套结构

举例1：

In [15]:
from pydantic import BaseModel
class Address(BaseModel):
    """地点描述"""
    city: str
    district: str
class Company(BaseModel):
    """公司信息"""
    name: str
    address: Address # 嵌套模型

structured_llm = model.with_structured_output(Company)
result = structured_llm.invoke("阿里巴巴在杭州滨江区")
print(result)

name='阿里巴巴' address=Address(city='杭州', district='滨江区')


举例2：

In [16]:
from pydantic import BaseModel, Field
from typing import List
# 1. 定义嵌套的 Pydantic 模型
class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    cast: List[Actor] = Field(description="演员列表") # 定义列表字段
    rating: float = Field(description="评分")

# 2. 初始化模型并绑定输出结构
structured_model = model.with_structured_output(Movie)
# 3. 调用模型，直接获取 Movie 实例
response = structured_model.invoke("请介绍电影《盗梦空间》")
# 4. 访问嵌套数据
print(f"电影名: {response.title}")
print(f"上映年份: {response.year}")
print(f"导演: {response.director}")
print(f"演员列表: {response.cast}")
print(f"评分: {response.rating}")

电影名: 盗梦空间
上映年份: 2010
导演: 克里斯托弗·诺兰
演员列表: [Actor(name='莱昂纳多·迪卡普里奥', role='多姆·柯布'), Actor(name='约瑟夫·戈登-莱维特', role='亚瑟'), Actor(name='艾伦·佩吉', role='阿里阿德妮'), Actor(name='汤姆·哈迪', role='伊姆斯'), Actor(name='渡边谦', role='斋藤')]
评分: 8.8


LLM 能力有限，复杂嵌套结构可能会出错。

所以建议：

嵌套层级 ≤ 3 层

## 1.6 限制条件

在 `Field` 里可以直接写约束条件，Pydantic 校验时会检查。这些约束也会写进 JSON Schema，随请求发给模型：

| 参数 | 含义 | 适用类型 | 对应的 JSON Schema 关键字 |
|---|---|---|---|
| `gt` / `ge` | 大于 / 大于等于 | 数字 | `exclusiveMinimum` / `minimum` |
| `lt` / `le` | 小于 / 小于等于 | 数字 | `exclusiveMaximum` / `maximum` |
| `multiple_of` | 必须是某个数的倍数 | 数字 | `multipleOf` |
| `min_length` / `max_length` | 最短 / 最长 | 字符串、列表 | 字符串：`minLength` / `maxLength`；列表：`minItems` / `maxItems` |
| `pattern` | 必须匹配的正则表达式 | 字符串 | `pattern` |

先看 Pydantic 本身的校验效果（不涉及大模型）：

In [17]:
from pydantic import ValidationError

class User(BaseModel):
    name: str = Field(min_length=2,max_length=20)
    age: int = Field(ge=0,le=150)
    email: str

print("\n有效数据")

try:
    user = User(name="张三", age=30, email="zhang@example.com")
    print(f"[OK] {user.name}, {user.age}, {user.email}")
except ValidationError as e:
    print(f"[FAIL] {e}")

print("\n无效数据（年龄超出范围）:")
try:
    user = User(name="李四", age=200, email="li@example.com")
    print(f"[OK] {user}")
except ValidationError as e:
    print(f"[FAIL] 验证失败（符合预期）: {e.errors()[0]['msg']}")


有效数据
[OK] 张三, 30, zhang@example.com

无效数据（年龄超出范围）:
[FAIL] 验证失败（符合预期）: Input should be less than or equal to 150


再把约束交给大模型。原文里的价格和库存都是负数，先用 GPT：

In [20]:
class Product(BaseModel):
    """产品信息（严格验证）"""
    name: str = Field(description="产品名称（字符串类型）", min_length=2)
    price: float = Field(description="价格，数字类型", gt=0)
    stock: int = Field(description="库存，整数类型", ge=0)

# 测试
structured_llm = model.with_structured_output(Product)
response = structured_llm.invoke("华为mate 80 promax 价格是-7999，当前库存-500")
print(response)

name='华为Mate 80 Pro Max' price=7999.0 stock=500


换成 DeepSeek：

In [21]:
structured_llm = model_deepseek.with_structured_output(Product)
try:
    response = structured_llm.invoke("华为mate 80 promax 价格是-7999，当前库存-500")
    print(response)
except ValidationError as e:
    print(e)

2 validation errors for Product
price
  Input should be greater than 0 [type=greater_than, input_value=-7999, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
stock
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-500, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal


GPT 返回了 7999 和 500，满足约束；DeepSeek 返回了 -7999 和 -500，被 Pydantic 校验拦了下来（报错信息里的 `input_value` 就是模型实际返回的值）。同一个类，为什么结果不同？原因见 1.7。

## 1.7 为什么不同厂商对默认值和限制条件的处理不同？

前面有两处，同一个 Pydantic 类交给 GPT 和 DeepSeek，结果不一样：

| 现象 | GPT（`model`，ChatOpenAI） | DeepSeek（`model_deepseek`） |
|---|---|---|
| 1.2 默认值：`age` 默认 32，原文没提年龄 | `age=0`，没有用默认值 | `age=32` |
| 1.6 限制条件：价格、库存不能为负，原文是负数 | `7999`、`500`，满足约束 | `-7999`、`-500`，校验报错 |

先说结论：**Pydantic 在本地做的事完全一样**。JSON 里缺了某个字段，就补上默认值；值不满足约束，就报错。不同的是**模型返回的 JSON**，而模型返回什么样的 JSON，取决于两件事：

1. **schema 走哪条路线发给模型**：由 `with_structured_output` 的 `method` 决定，不同厂商的 LangChain 集成默认值不同
2. **服务端是否强制模型遵守 schema**：也就是有没有开启 strict 模式，以及服务端支持哪些 JSON Schema 关键字

### 1.7.1 同一个类，两条路线

上一节讲过，`ChatOpenAI` 默认 `method="json_schema"`：OpenAI SDK 把 Pydantic 类转成 **strict** 的 JSON Schema，放进请求的 `response_format`。

`ChatDeepSeek`（`init_chat_model(..., model_provider="deepseek")` 创建的就是它）默认 `method="function_calling"`：把 schema 当成一个工具，并且**不开启 strict**。DeepSeek 的接口不支持 `json_schema`，即使传 `method="json_schema"`，`ChatDeepSeek` 也会自动改成 `function_calling`。

把 1.2 的 `Person` 按两条路线各转换一次（不请求模型）：

In [22]:
import json
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.utils.function_calling import convert_to_openai_tool
from openai.lib._parsing._completions import type_to_response_format_param

class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(default=32, description="年龄")
    occupation: str = Field(description="职业")

# GPT 路线：ChatOpenAI 的 json_schema，由 OpenAI SDK 转换（上一节 2.1.2 用过这个函数）
gpt_format = type_to_response_format_param(Person)["json_schema"]
# DeepSeek 路线：ChatDeepSeek 的 function_calling，把 Person 当成工具
deepseek_tool = convert_to_openai_tool(Person)["function"]

print("GPT      strict =", gpt_format["strict"], "| required =", gpt_format["schema"]["required"])
print("DeepSeek strict =", deepseek_tool.get("strict"), "| required =", deepseek_tool["parameters"]["required"])
print()
print("age 字段（两边相同）:", json.dumps(deepseek_tool["parameters"]["properties"]["age"], ensure_ascii=False))

GPT      strict = True | required = ['name', 'age', 'occupation']
DeepSeek strict = None | required = ['name', 'occupation']

age 字段（两边相同）: {"anyOf": [{"type": "integer"}, {"type": "null"}], "default": 32, "description": "年龄"}


| | GPT（ChatOpenAI） | DeepSeek（ChatDeepSeek） |
|---|---|---|
| 默认 `method` | `json_schema` | `function_calling` |
| schema 放在请求的哪里 | `response_format` | `tools`（并强制模型调用这个工具） |
| strict | 开启（OpenAI SDK 自动设置） | 未开启 |
| `required` | **所有字段**，包括有默认值的 `age` | 只有没有默认值的字段 |

### 1.7.2 默认值：strict 模式要求所有字段必填

OpenAI 文档对 strict 模式的要求是：**所有字段都必须是必填的，模型会为每个字段返回一个值**；想表达"可选"，只能用"值或 null"，也就是 `Optional`。DeepSeek 的 strict 模式（Beta）也有同样的要求。

所以默认值能不能生效，取决于走哪条路线：

- **GPT**：`age` 被放进了 `required`，模型**必须**给它一个值。原文没提年龄，模型只能自己填一个，这次填了 0（也可能填 null）。JSON 里有 `age`，Pydantic 就不会用默认值
- **DeepSeek**：`age` 不在 `required` 里，模型可以不输出它。JSON 里缺了 `age`，Pydantic 才会补上默认值 32

用 `include_raw=True` 看看 DeepSeek 实际返回的工具参数（会请求一次 DeepSeek）：

In [23]:
raw_result = model_deepseek.with_structured_output(Person, include_raw=True).invoke("张三是一名医生")

print("模型返回的参数:", raw_result["raw"].tool_calls[0]["args"])
print("解析后的对象:  ", repr(raw_result["parsed"]))

模型返回的参数: {'name': '张三', 'occupation': '医生'}
解析后的对象:   Person(name='张三', age=32, occupation='医生')


模型返回的参数里**根本没有 `age`**，32 是 Pydantic 在本地补上的。

**Pydantic 的默认值只在"JSON 里缺这个字段"时生效。** 1.3 的 `email: Optional[str] = Field("未提供", ...)` 也是同样的情况：GPT 必须给 `email` 一个值，它给了 null，所以打印出来是 `None`，而不是"未提供"。

### 1.7.3 限制条件：谁来执行约束

限制条件的情况不太一样：`gt=0`、`ge=0`、`min_length=2` 在两条路线的 schema 里**都在**，分别是 `exclusiveMinimum`、`minimum`、`minLength`。区别在于**有没有人强制执行**：

- **GPT（strict）**：OpenAI 文档列出，strict 模式支持 `minimum`、`exclusiveMinimum` 等数值约束，模型生成时就受到限制，不能输出负数，所以返回了 7999 和 500。（你用的是中转服务，约束最终在哪一步执行，我们从外面看不到，但结果与此一致。）
- **DeepSeek（未开启 strict）**：schema 只是一份"说明"，模型可以不遵守。它忠实地提取了原文的 -7999 和 -500，到了本地被 Pydantic 校验拦下，抛出 `ValidationError`。报错信息里的 `input_value=-7999`，就是模型实际返回的值

另外，各厂商在 strict 模式下**支持的约束关键字也不一样**。根据两家的官方文档：

| Pydantic 写法 | JSON Schema 关键字 | OpenAI strict | DeepSeek strict（Beta） |
|---|---|---|---|
| `gt` / `ge` / `lt` / `le` / `multiple_of` | `exclusiveMinimum` / `minimum` / `exclusiveMaximum` / `maximum` / `multipleOf` | 支持 | 支持 |
| 字符串的 `pattern` | `pattern` | 支持 | 支持 |
| 字符串的 `min_length` / `max_length` | `minLength` / `maxLength` | 支持列表中未明确列出 | 不支持 |
| 列表的 `min_length` / `max_length` | `minItems` / `maxItems` | 支持 | 不支持 |

两家文档都说明：开启 strict 后，如果 schema 里有服务端不支持的写法，接口会直接返回错误。所以同一个类换厂商时，开 strict 也不一定能直接用。

**注意：GPT 返回的 7999 并不是"更正确"。** 原文写的是 -7999，strict 模式为了满足 schema，把负号"吞"掉了，数据被悄悄改了；DeepSeek 忠实地提取了 -7999，由 Pydantic 报错，把问题暴露了出来。约束解码只保证**格式合法**，不保证**内容真实**。所以提取数据时，约束更适合用来"发现异常"，而不是"修正数据"。

### 1.7.4 应对建议

| 做法 | 原因 |
|---|---|
| 把本地 Pydantic 校验当作最后一道防线，做好 `ValidationError` 的处理 | 不是每个厂商都会在生成时执行约束 |
| 用 `include_raw=True` 查看模型的原始输出 | 校验失败时能看到模型到底返回了什么；`function_calling` 路线下还能兜住校验异常（见上一节 2.1.4） |
| 表达"原文没提到"时，用 `Optional` 让模型返回 null，默认值在本地补 | strict 路线下 `Field(default=...)` 不会生效；本地补的写法见 1.8.1 |
| 换厂商时，先用几条样例测一测 | 同一个类，在不同厂商上的表现可能不同 |

## 1.8 其他常用的高级特性

除了前面 6 种情况，下面三个特性在结构化输出中也很常用：

| 特性 | 解决什么问题 |
|---|---|
| 自定义校验器 `field_validator` / `model_validator` | `Field` 的约束只能表达范围、长度、格式，更复杂的规则和数据清洗交给校验器 |
| `Union` 联合类型 | 同一个输入可能对应不同的结构，让模型选一种 |
| `model_dump` / `model_dump_json` | 把结果转成字典或 JSON，交给后续程序 |

### 1.8.1 自定义校验器：field_validator 与 model_validator

`Field` 的约束只能表达"范围、长度、格式"这类简单规则。更复杂的规则，比如"把手机号统一成 11 位数字""返回日期不能早于出发日期"，可以写成校验器：

- `@field_validator("字段名")`：校验单个字段。既可以**清洗**数据（返回新值），也可以**拒绝**数据（抛出 `ValueError`）
- `@model_validator(mode="after")`：所有字段都解析完之后运行，用来检查**多个字段之间的关系**

两个要点：

1. 校验器是 Python 代码，**不会出现在 JSON Schema 里，模型看不到**。它只在本地解析时运行
2. `with_structured_output` 拿到模型的回复后，最终都要用 Pydantic 创建对象（GPT 路线调用 `model_validate_json`，DeepSeek 路线调用构造函数），所以校验器会自动生效

下面直接用 `model_validate_json` 模拟"模型返回了这段 JSON"，不请求模型：

In [24]:
from pydantic import field_validator, model_validator

class Contact(BaseModel):
    """联系人信息"""
    name: str = Field(description="姓名")
    phone: str = Field(description="手机号，11 位数字")

    @field_validator("phone")
    @classmethod
    def normalize_phone(cls, v: str) -> str:
        digits = "".join(ch for ch in v if ch.isdigit())   # 去掉空格、横线等分隔符
        if len(digits) != 11:
            raise ValueError(f"手机号应为 11 位数字，实际是：{v}")
        return digits                                      # 返回清洗后的值

# 模型返回的手机号带横线：校验器把它清洗成统一格式
print(repr(Contact.model_validate_json('{"name": "王小明", "phone": "138-1234-5678"}')))

# 位数不对：校验器拒绝
try:
    Contact.model_validate_json('{"name": "王小明", "phone": "138-1234"}')
except ValidationError as e:
    print("ValidationError ->", e.errors()[0]["msg"])

# 校验器不会出现在 schema 里，模型只能看到 description
print(Contact.model_json_schema()["properties"]["phone"])

Contact(name='王小明', phone='13812345678')
ValidationError -> Value error, 手机号应为 11 位数字，实际是：138-1234
{'description': '手机号，11 位数字', 'title': 'Phone', 'type': 'string'}


`model_validator` 检查多个字段之间的关系。这里顺便用到了 `date` 类型：模型返回的是 `"2026-10-01"` 这样的字符串，Pydantic 会把它转换成 Python 的 `date` 对象。

In [25]:
from datetime import date

class Trip(BaseModel):
    """行程信息"""
    start: date = Field(description="出发日期")
    end: date = Field(description="返回日期")

    @model_validator(mode="after")
    def check_dates(self):
        if self.end < self.start:
            raise ValueError("返回日期不能早于出发日期")
        return self

print(repr(Trip.model_validate_json('{"start": "2026-10-01", "end": "2026-10-07"}')))

try:
    Trip.model_validate_json('{"start": "2026-10-01", "end": "2026-09-28"}')
except ValidationError as e:
    print("ValidationError ->", e.errors()[0]["msg"])

Trip(start=datetime.date(2026, 10, 1), end=datetime.date(2026, 10, 7))
ValidationError -> Value error, 返回日期不能早于出发日期


回到 1.7.2 的问题：strict 路线下 `Field(default=...)` 不生效，可以改成"让模型返回 null，再用校验器补上默认值"。这样两条路线的结果就一致了：

In [26]:
class PersonWithDefault(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄，原文没有提到则为 null")
    occupation: str = Field(description="职业")

    @field_validator("age")
    @classmethod
    def fill_default_age(cls, v):
        return 32 if v is None else v   # 模型返回 null 时，在本地补上默认值

print(repr(PersonWithDefault.model_validate_json('{"name": "张三", "age": null, "occupation": "医生"}')))

PersonWithDefault(name='张三', age=32, occupation='医生')


### 1.8.2 Union：让模型在多种结构中选一种

有时同一个入口会收到不同类型的内容。比如客服消息可能是投诉，也可能是咨询，需要提取的字段完全不同。用 `Union[A, B]` 把几种结构都列出来，让模型选一种：

- 每种结构加一个 `Literal` 类型的 `kind` 字段，标明自己是哪一种，方便模型选择，也方便 Pydantic 区分
- `Union` 要放在一个外层模型的字段里。OpenAI 要求 schema 的最外层必须是对象，不能直接是"几选一"（`anyOf`）

下面用 `batch` 一次处理两条消息（上一节 2.1.3 提到的 Runnable 方法），会请求两次模型：

In [27]:
from typing import Union, Literal

class Complaint(BaseModel):
    """投诉"""
    kind: Literal["complaint"] = Field(description="消息类型，固定为 complaint")
    order_id: Optional[str] = Field(description="订单号，没有提到则为 null")
    problem: str = Field(description="投诉的问题")

class Inquiry(BaseModel):
    """咨询"""
    kind: Literal["inquiry"] = Field(description="消息类型，固定为 inquiry")
    question: str = Field(description="咨询的问题")

class CustomerMessage(BaseModel):
    """客户消息的分类结果"""
    result: Union[Complaint, Inquiry] = Field(description="投诉或咨询，二选一")

structured_llm = model.with_structured_output(CustomerMessage)
messages = [
    "我的订单 A1024 三天了还没发货，太慢了！",
    "你们的会员可以同时在几台设备上登录？",
]
for text, msg in zip(messages, structured_llm.batch(messages)):
    print(f"{text}\n  -> {type(msg.result).__name__}: {msg.result}\n")

我的订单 A1024 三天了还没发货，太慢了！
  -> Complaint: kind='complaint' order_id='A1024' problem='订单三天了还没发货，发货太慢。'

你们的会员可以同时在几台设备上登录？
  -> Inquiry: kind='inquiry' question='你们的会员可以同时在几台设备上登录？'



两条消息分别得到了 `Complaint` 和 `Inquiry` 对象，后续代码可以根据 `kind` 字段或 `isinstance` 分别处理。

### 1.8.3 转成字典或 JSON：model_dump

结构化输出的结果通常要交给后续程序，比如作为接口的返回值、写进数据库、传给下一条链。Pydantic 对象可以很方便地转成字典或 JSON 字符串：

In [28]:
person = Person(name="张三", occupation="医生")   # 在本地创建对象时，默认值正常生效

print(person.model_dump())                  # 转成字典
print(person.model_dump(exclude={"age"}))   # 排除某些字段
print(person.model_dump_json())             # 转成 JSON 字符串

trip = Trip(start="2026-10-01", end="2026-10-07")
print(trip.model_dump())                    # 日期保持为 date 对象
print(trip.model_dump(mode="json"))         # mode="json"：转成 JSON 能表示的类型，日期变成字符串

{'name': '张三', 'age': 32, 'occupation': '医生'}
{'name': '张三', 'occupation': '医生'}
{"name":"张三","age":32,"occupation":"医生"}
{'start': datetime.date(2026, 10, 1), 'end': datetime.date(2026, 10, 7)}
{'start': '2026-10-01', 'end': '2026-10-07'}


## 1.9 总结

| 需求 | 写法 | 注意 |
|---|---|---|
| 字段可能没有 | `Optional[类型]` | 模型会返回 null |
| 没有时用默认值 | `Field(default=...)` | strict 路线（如 GPT）下不生效，见 1.7.2 |
| 限定可选值 | `Enum` / `Literal` | `Literal` 写起来简单，`Enum` 方便复用 |
| 多个同类对象 | `List[模型]` | |
| 嵌套结构 | 字段类型写成另一个模型 | 建议不超过 3 层 |
| 数值范围、长度、格式 | `Field(gt=..., min_length=..., pattern=...)` | 各厂商的执行程度不同，见 1.7.3 |
| 复杂规则、数据清洗 | `field_validator` / `model_validator` | 只在本地运行，模型看不到 |
| 多种结构选一种 | `Union[A, B]` | 放在外层模型的字段里 |
| 转成字典或 JSON | `model_dump` / `model_dump_json` | |

**一句话记住 1.7：** Pydantic 在本地的行为处处一样，不同的是模型返回的 JSON；而 JSON 长什么样，取决于 `method` 和服务端是否开启 strict。

下一节讲 Pydantic 的工作流程，把本节多次提到的"本地解析和校验"完整地串起来。